In [6]:
import numpy as np
import pandas as pd
import tkinter as tk
from tkinter import messagebox
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
import warnings

warnings.filterwarnings("ignore")

# =========================================================
# 1. 数据
# =========================================================
data = pd.read_excel("dataset #1.xlsx")
data = pd.DataFrame(data)

# =========================================================
# 2. 模型1：SCOD & VFAs
# =========================================================
X_scod = data.values[:, 0:8]
Y_scod = data.values[:, 8:10]

X1_train, _, y1_train, _ = train_test_split(
    X_scod, Y_scod, test_size=0.2, random_state=42
)

model_scod_vfas = RandomForestRegressor(
    n_estimators=186,
    max_depth=121,
    min_samples_split=9,
    min_samples_leaf=7,
    max_features='sqrt',
    bootstrap=False,
    random_state=42
)
model_scod_vfas.fit(X1_train, y1_train)

# =========================================================
# 3. 模型2：MY & MPR
# =========================================================
X_final = data.values[:, 0:15]
Y_final = data.values[:, 15:17]

X2_train, _, y2_train, _ = train_test_split(
    X_final, Y_final, test_size=0.2, random_state=42
)

model_final = RandomForestRegressor(
    n_estimators=289,
    max_depth=173,
    min_samples_split=3,
    min_samples_leaf=2,
    max_features="sqrt",
    bootstrap=False,
    random_state=42
)
model_final.fit(X2_train, y2_train)

# =========================================================
# 4. 格式化
# =========================================================
def fmt(x):
    return "" if pd.isna(x) else f"{x:.2f}"

# =========================================================
# 5. 输入范围
# =========================================================
input_ranges = {}
all_X = data.values[:, 0:15]

mapping = {
    "Cellulose": 0,
    "Hemicellulose": 1,
    "Lignin": 2,
    "TS": 3,
    "VS": 4,
    "SC": 5,
    "HTP-T": 6,
    "HTP-RT": 7,
    "AD-T": 13,
    "AD-Time": 14
}

for k, i in mapping.items():
    input_ranges[k] = f"[{fmt(np.min(all_X[:, i]))}, {fmt(np.max(all_X[:, i]))}]"

# =========================================================
# 6. GUI 基础设置
# =========================================================
root = tk.Tk()
root.title("AD performance prediction software")

root.geometry("1630x860")
root.minsize(750, 700)

root.resizable(True, True)

BG = "#f2f2f2"
root.configure(bg=BG)

FONT = ("Times New Roman", 20)
FONT_B = ("Times New Roman", 20, "bold")

entries = {}

status = tk.StringVar(value="Ready")

scod_var = tk.StringVar()
vfas_var = tk.StringVar()

ratio_var = tk.StringVar()
scod_dil_var = tk.StringVar()
vfas_dil_var = tk.StringVar()

my_var = tk.StringVar()
mpr_var = tk.StringVar()

# =========================================================
# 7. 布局参数
# =========================================================

LEFT_PANEL_X = 15
LEFT_PANEL_Y = 15
LEFT_PANEL_W = 800
LEFT_PANEL_H = 650

RIGHT_PANEL_X = 815
RIGHT_PANEL_Y = 15
RIGHT_PANEL_W = 800
RIGHT_PANEL_H = 650

# -------- 左侧 --------
LEFT_BIO_X = 0
LEFT_BIO_Y = 0
LEFT_BIO_W = 770
LEFT_BIO_H = 390

LEFT_HTP_X = 0
LEFT_HTP_Y = 405
LEFT_HTP_W = 770
LEFT_HTP_H = 200

LEFT_AD_X = 0
LEFT_AD_Y = 540
LEFT_AD_W = 770
LEFT_AD_H = 120

# -------- 右侧 --------
RIGHT_FEED_X = 0
RIGHT_FEED_Y = 0
RIGHT_FEED_W = 770
RIGHT_FEED_H = 390

RIGHT_AD_X = 0
RIGHT_AD_Y = 405
RIGHT_AD_W = 770
RIGHT_AD_H = 200

# -------- 底部 --------
BOTTOM_MY_MPR_X = 15
BOTTOM_MY_MPR_Y = 625
BOTTOM_MY_MPR_W = 1570
BOTTOM_MY_MPR_H = 160

BOTTOM_BUTTONS_Y = 810

# -------- SCOD 按钮 --------
SCOD_BTN_X = 360
SCOD_BTN_Y = 315

# -------- 通用尺寸 --------
COMMON_ENTRY_WIDTH = 20

COMMON_ROW_PADY = 8

COMMON_LABEL_COL_W = 300
COMMON_ENTRY_COL_W = 220
COMMON_RANGE_COL_W = 220

# =========================================================
# 8. 输入 / 输出组件
# =========================================================
def add_input(p, text, key, row):

    p.grid_columnconfigure(0, minsize=COMMON_LABEL_COL_W)
    p.grid_columnconfigure(1, minsize=COMMON_ENTRY_COL_W)
    p.grid_columnconfigure(2, minsize=COMMON_RANGE_COL_W)

    tk.Label(
        p,
        text=text,
        bg=BG,
        font=FONT
    ).grid(
        row=row,
        column=0,
        sticky="w",
        padx=10,
        pady=COMMON_ROW_PADY
    )

    e = tk.Entry(
        p,
        width=COMMON_ENTRY_WIDTH,
        font=FONT,
        bg="white",
        relief="solid"
    )

    e.grid(
        row=row,
        column=1,
        sticky="nw"
    )

    tk.Label(
        p,
        text=input_ranges[key],
        bg=BG,
        fg="#666",
        font=FONT
    ).grid(
        row=row,
        column=2,
        sticky="nw"
    )

    entries[key] = e


def add_output(p, text, var, row):

    p.grid_columnconfigure(0, minsize=COMMON_LABEL_COL_W)
    p.grid_columnconfigure(1, minsize=COMMON_ENTRY_COL_W)

    tk.Label(
        p,
        text=text,
        bg=BG,
        font=FONT
    ).grid(
        row=row,
        column=0,
        sticky="w",
        padx=10,
        pady=COMMON_ROW_PADY
    )

    tk.Entry(
        p,
        textvariable=var,
        state="readonly",
        bg="#e6e6e6",
        width=COMMON_ENTRY_WIDTH,
        font=FONT,
        relief="sunken",
        readonlybackground="#e6e6e6"
    ).grid(
        row=row,
        column=1,
        sticky="nw"
    )

# =========================================================
# 9. 输入读取
# =========================================================
def get_x8():

    return np.array([
        float(entries["Cellulose"].get()),
        float(entries["Hemicellulose"].get()),
        float(entries["Lignin"].get()),
        float(entries["TS"].get()),
        float(entries["VS"].get()),
        float(entries["SC"].get()),
        float(entries["HTP-T"].get()),
        float(entries["HTP-RT"].get())
    ]).reshape(1, -1)


def get_ad():

    return (
        float(entries["AD-T"].get()),
        float(entries["AD-Time"].get())
    )

# =========================================================
# 10. SCOD / VFAs 预测
# =========================================================
def predict_scod_vfas():

    try:
        x = get_x8()

        scod, vfas = model_scod_vfas.predict(x)[0]

        scod_var.set(fmt(scod))
        vfas_var.set(fmt(vfas))

        status.set("SCOD and VFAs done")

    except:
        messagebox.showerror("Error", "Invalid input")

# =========================================================
# 11. MY / MPR 预测
# =========================================================
def predict_my_mpr():

    try:
        x8 = get_x8()

        ad_t, ad_time = get_ad()

        scod, vfas = model_scod_vfas.predict(x8)[0]

        r_in = ratio_entry.get().strip()

        ratios = [float(r_in)] if r_in != "" else np.arange(1.0, 100.1, 0.1)

        best_my = -1e9

        for r in ratios:

            scod_d = scod / r
            vfas_d = vfas / r

            X = np.array([
                *x8.flatten(),
                scod,
                vfas,
                r,
                scod_d,
                vfas_d,
                ad_t,
                ad_time
            ]).reshape(1, -1)

            my, mpr = model_final.predict(X)[0]

            if my > best_my:

                best_my = my
                best_mpr = mpr
                best_r = r
                best_sd = scod_d
                best_vd = vfas_d

        ratio_var.set(fmt(best_r))
        scod_dil_var.set(fmt(best_sd))
        vfas_dil_var.set(fmt(best_vd))

        my_var.set(fmt(best_my))
        mpr_var.set(fmt(best_mpr))

        status.set("MY and MPR done")

    except:
        messagebox.showerror("Error", "Invalid input")

# =========================================================
# 12. 清除
# =========================================================
def clear_all():

    for e in entries.values():
        e.delete(0, tk.END)

    ratio_entry.delete(0, tk.END)

    scod_var.set("")
    vfas_var.set("")

    ratio_var.set("")
    scod_dil_var.set("")
    vfas_dil_var.set("")

    my_var.set("")
    mpr_var.set("")

    status.set("Ready")

# =========================================================
# 13. 主布局
# =========================================================
main = tk.Frame(root, bg=BG)
main.pack(fill="both", expand=True)

# =========================================================
# 左侧面板
# =========================================================
left_panel = tk.Frame(main, bg=BG)

left_panel.place(
    x=LEFT_PANEL_X,
    y=LEFT_PANEL_Y,
    width=LEFT_PANEL_W,
    height=LEFT_PANEL_H
)

# =========================================================
# 右侧面板
# =========================================================
right_panel = tk.Frame(main, bg=BG)

right_panel.place(
    x=RIGHT_PANEL_X,
    y=RIGHT_PANEL_Y,
    width=RIGHT_PANEL_W,
    height=RIGHT_PANEL_H
)

# =========================================================
# 左：生物质性质
# =========================================================
bio = tk.LabelFrame(
    left_panel,
    text="Lignocellulosic biomass characteristics",
    font=FONT_B,
    bg=BG
)

bio.place(
    x=LEFT_BIO_X,
    y=LEFT_BIO_Y,
    width=LEFT_BIO_W,
    height=LEFT_BIO_H
)

bio_inner = tk.Frame(bio, bg=BG)
bio_inner.pack(fill="both", expand=True)

tk.Label(
    bio_inner,
    text="Biochemical composition",
    bg=BG,
    font=FONT_B
).grid(row=0, column=0, sticky="w")

add_input(bio_inner, "Cellulose (%)", "Cellulose", 1)
add_input(bio_inner, "Hemicellulose (%)", "Hemicellulose", 2)
add_input(bio_inner, "Lignin (%)", "Lignin", 3)

tk.Label(
    bio_inner,
    text="Solid properties",
    bg=BG,
    font=FONT_B
).grid(row=4, column=0, sticky="w")

add_input(bio_inner, "TS (%)", "TS", 5)
add_input(bio_inner, "VS (%)", "VS", 6)

# =========================================================
# 左：HTP
# =========================================================
htp = tk.LabelFrame(
    left_panel,
    text="HTP conditions",
    font=FONT_B,
    bg=BG
)

htp.place(
    x=LEFT_HTP_X,
    y=LEFT_HTP_Y,
    width=LEFT_HTP_W,
    height=LEFT_HTP_H
)

add_input(htp, "SC (%)", "SC", 0)
add_input(htp, "HTP-T (℃)", "HTP-T", 1)
add_input(htp, "HTP-RT (min)", "HTP-RT", 2)

# =========================================================
# 右：AD Feed
# =========================================================
feed = tk.LabelFrame(
    right_panel,
    text="AD Feed Properties",
    font=FONT_B,
    bg=BG
)

feed.place(
    x=RIGHT_FEED_X,
    y=RIGHT_FEED_Y,
    width=RIGHT_FEED_W,
    height=RIGHT_FEED_H
)

feed_inner = tk.Frame(feed, bg=BG)
feed_inner.pack(fill="both", expand=True)

feed_inner.grid_columnconfigure(0, minsize=COMMON_LABEL_COL_W)
feed_inner.grid_columnconfigure(1, minsize=COMMON_ENTRY_COL_W)
feed_inner.grid_columnconfigure(2, minsize=COMMON_RANGE_COL_W)

add_output(feed_inner, "SCOD (mg/L)", scod_var, 0)
add_output(feed_inner, "VFAs (mg/L)", vfas_var, 1)

tk.Label(
    feed_inner,
    text="Dilution ratio",
    bg=BG,
    font=FONT
).grid(
    row=2,
    column=0,
    sticky="w",
    padx=10,
    pady=COMMON_ROW_PADY
)

ratio_entry = tk.Entry(
    feed_inner,
    width=COMMON_ENTRY_WIDTH,
    font=FONT,
    bg="white",
    relief="solid"
)

ratio_entry.grid(
    row=2,
    column=1,
    sticky="nw"
)

tk.Label(
    feed_inner,
    text="[1.00, 73.00]",
    bg=BG,
    fg="#666",
    font=FONT
).grid(
    row=2,
    column=2,
    sticky="nw"
)

add_output(feed_inner, "Auto dilution ratio", ratio_var, 3)
add_output(feed_inner, "SCOD_dil (mg/L)", scod_dil_var, 4)
add_output(feed_inner, "VFAs_dil (mg/L)", vfas_dil_var, 5)

# =========================================================
# SCOD按钮
# =========================================================
btn_scod = tk.Frame(feed, bg=BG)

btn_scod.place(
    x=SCOD_BTN_X,
    y=SCOD_BTN_Y
)

tk.Button(
    btn_scod,
    text="Predict SCOD and VFAs",
    width=20,
    command=predict_scod_vfas
).pack()

# =========================================================
# 右：AD 参数
# =========================================================
ad_right = tk.LabelFrame(
    right_panel,
    text="AD Parameters",
    font=FONT_B,
    bg=BG
)

ad_right.place(
    x=RIGHT_AD_X,
    y=RIGHT_AD_Y,
    width=RIGHT_AD_W,
    height=RIGHT_AD_H
)

add_input(ad_right, "AD-T (℃)", "AD-T", 0)
add_input(ad_right, "AD-Time (d)", "AD-Time", 1)

# =========================================================
# 底部：AD performance
# =========================================================
final = tk.LabelFrame(
    main,
    text="AD performance",
    font=FONT_B,
    bg=BG
)

final.place(
    x=BOTTOM_MY_MPR_X,
    y=BOTTOM_MY_MPR_Y,
    width=BOTTOM_MY_MPR_W,
    height=BOTTOM_MY_MPR_H
)

final_inner = tk.Frame(final, bg=BG)
final_inner.pack(fill="both", expand=True)

add_output(final_inner, "MY (mL/g VS)", my_var, 0)
add_output(final_inner, "MPR (mL/(g VS·d))", mpr_var, 1)

# =========================================================
# 底部按钮
# =========================================================
btn = tk.Frame(main, bg=BG)

btn.place(
    x=0,
    y=BOTTOM_BUTTONS_Y,
    width=1600,
    height=60
)

center_buttons = tk.Frame(btn, bg=BG)
center_buttons.pack(anchor="center")

tk.Button(
    center_buttons,
    text="Predict MY and MPR",
    width=20,
    command=predict_my_mpr
).pack(side="left", padx=10)

tk.Button(
    center_buttons,
    text="Clear",
    width=20,
    command=clear_all
).pack(side="left", padx=10)

# =========================================================
# 状态栏
# =========================================================
tk.Label(
    main,
    textvariable=status,
    bg=BG
).place(
    x=15,
    y=840,
    width=1570,
    height=25
)

root.mainloop()